# Phase 1 — Combined Binary Classification Baselines
**Runtime:** Google Colab — **GPU (T4 or A100)**

**Purpose:** Unified notebook combining NB02 (XGBoost), NB03 (MOSAIKS Ridge),
NB04 (ResNet-18 LogReg), NB05 (DINOv2 LogReg), and NB07 (Prithvi linear probe)
into a single run. Data is loaded **once** and shared across all five models.

| # | Model | Features | Classifier |
|---|---|---|---|
| 1 | XGBoost Engineered | 35-d hand-crafted | XGBClassifier |
| 2 | MOSAIKS + Ridge | 4096-d random conv | RidgeClassifierCV |
| 3 | ResNet-18 + LogReg | 512-d frozen backbone | LogisticRegressionCV |
| 4 | DINOv2 + LogReg | 768-d frozen CLS token | LogisticRegressionCV |
| 5 | Prithvi-300M Linear | 1024-d frozen encoder | LayerNorm→Dropout→Linear (Lightning) |

**Task:** Binary classification — `has_coverage` from `label_confidence == 'confirmed_positive'`

**Evaluation:** Val-optimal threshold via PR-curve F1 maximisation, then test-set metrics:
PR-AUC, ROC-AUC, F1, Precision, Recall, Accuracy, Confusion Matrix, PR Curve.

**Inputs:**
- `data/raw/patches_2019/` — 6,970 GeoTIFF patches (224×224×6 HLS bands)
- `data/processed/sampled_patches.csv` — patch metadata + labels

## Step 0: Environment Setup

In [ ]:
# ============================================================
# Cell 0.1: Install dependencies (includes terratorch for Prithvi)
# Pin numpy BEFORE terratorch so it cannot be upgraded to 2.x
# ============================================================
!pip install -q "numpy==1.26.4"
!pip install -q terratorch
!pip install -q xgboost rasterio scikit-learn matplotlib seaborn

# Restart runtime so all C extensions load against the pinned numpy
import os
print("Restarting runtime to apply numpy pin — re-run from Cell 0.2 after restart.")
os.kill(os.getpid(), 9)

In [ ]:
# ============================================================
# Cell 0.2: Clone repo
# ============================================================
import os

REPO = 'satellite-internet-prediction'
if not os.getcwd().endswith(REPO):
    if os.path.exists(REPO):
        %cd {REPO}
    else:
        !git clone https://github.com/tatyana21111/{REPO}.git
        %cd {REPO}
!git pull

In [ ]:
# ============================================================
# Cell 0.3: Sync patches from Google Drive
# ============================================================
from google.colab import drive
import shutil
from pathlib import Path

drive.mount('/content/drive', force_remount=False)

PATCH_DIR = 'data/raw/patches_2019'
Path(PATCH_DIR).mkdir(parents=True, exist_ok=True)

local_count = len(list(Path(PATCH_DIR).glob('*.tif')))
print(f'Local patches: {local_count}')

if local_count < 6900:
    print('Syncing from Google Drive ...')
    drive_dir = '/content/drive/MyDrive/patches_2019'
    for f in Path(drive_dir).glob('*.tif'):
        dst = Path(PATCH_DIR) / f.name
        if not dst.exists():
            shutil.copy(f, dst)
    local_count = len(list(Path(PATCH_DIR).glob('*.tif')))
    print(f'Patches after sync: {local_count}')
else:
    print('Patches already available locally.')

## Step 1: Imports & Constants

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as Func
import torchvision.models as models
import pytorch_lightning as pl
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import logging
import subprocess
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.linear_model import RidgeClassifierCV, LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    average_precision_score, precision_recall_curve,
    precision_score, recall_score, confusion_matrix,
    classification_report,
)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
logging.getLogger('rasterio').setLevel(logging.ERROR)

# ── Deterministic seeding ────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass

HLS_MEANS = [0.14245495, 0.13921481, 0.12434631, 0.31420089, 0.20743526, 0.12046503]
HLS_STDS  = [0.04036231, 0.04186983, 0.05267646, 0.08222210, 0.06834774, 0.05294205]
SCALE     = 0.0001
BAND_NAMES = ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']

PATCH_DIR   = 'data/raw/patches_2019'
RESULTS_DIR = Path('outputs/results')
FIGURES_DIR = Path('outputs/figures')
MODELS_DIR  = Path('outputs/models')
for p in [RESULTS_DIR, FIGURES_DIR, MODELS_DIR]:
    p.mkdir(parents=True, exist_ok=True)
Path('checkpoints').mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## Step 2: Load Data & Train / Val / Test Split
Pre-computed by NB01 (`patches_with_splits.csv`): binary labels,
aggregate targets, stratification features, and geographic split.

In [ ]:
df = pd.read_csv('data/processed/patches_with_splits.csv')

# Filter to patches with available TIF files
available = set(f.stem for f in Path(PATCH_DIR).glob('*.tif'))
df = df[df['patch_id'].isin(available)].reset_index(drop=True)

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df   = df[df['split'] == 'val'].reset_index(drop=True)
test_df  = df[df['split'] == 'test'].reset_index(drop=True)

y_train = train_df['has_coverage'].values
y_val   = val_df['has_coverage'].values
y_test  = test_df['has_coverage'].values

print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test (NE): {len(test_df):,}')
print(f'Train positive: {train_df["has_coverage"].mean():.2%}')
print(f'Val   positive: {val_df["has_coverage"].mean():.2%}')
print(f'Test  positive: {test_df["has_coverage"].mean():.2%}')

## Step 4: Dataset Classes

In [ ]:
class PatchDataset(Dataset):
    """Loads 6-band HLS patches with standard normalization."""
    def __init__(self, df, patch_dir, n_bands=6):
        self.df        = df.reset_index(drop=True)
        self.patch_dir = patch_dir
        self.n_bands   = n_bands

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = f"{self.patch_dir}/{row['patch_id']}.tif"
        with rasterio.open(path) as src:
            img = src.read(list(range(1, self.n_bands + 1))).astype(np.float32)
        img *= SCALE
        for b in range(self.n_bands):
            img[b] = (img[b] - HLS_MEANS[b]) / HLS_STDS[b]
        img   = np.nan_to_num(img, nan=0.0)
        label = int(row['has_coverage'])
        return torch.from_numpy(img), label


class PrithviPatchDataset(Dataset):
    """Adds temporal dimension T=1 for Prithvi: (C, H, W) -> (C, T=1, H, W)."""
    def __init__(self, df, patch_dir, n_bands=6):
        self.base = PatchDataset(df, patch_dir, n_bands)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, label = self.base[idx]
        return img.unsqueeze(1), label


train_dataset = PatchDataset(train_df, PATCH_DIR)
val_dataset   = PatchDataset(val_df,   PATCH_DIR)
test_dataset  = PatchDataset(test_df,  PATCH_DIR)

prithvi_train_dataset = PrithviPatchDataset(train_df, PATCH_DIR)
prithvi_val_dataset   = PrithviPatchDataset(val_df,   PATCH_DIR)
prithvi_test_dataset  = PrithviPatchDataset(test_df,  PATCH_DIR)

print(f'Datasets — train: {len(train_dataset)}, val: {len(val_dataset)}, test: {len(test_dataset)}')
img, lbl = train_dataset[0]
print(f'PatchDataset sample shape: {img.shape}')
img_p, _ = prithvi_train_dataset[0]
print(f'PrithviPatchDataset sample shape: {img_p.shape}')

## Step 5A: Engineered Features (35-d) — for XGBoost

In [ ]:
def extract_engineered_features(patch_path):
    """Hand-crafted spectral + spatial features from a single patch (35 features)."""
    with rasterio.open(patch_path) as src:
        img = src.read().astype(np.float32) * SCALE

    feats = {}
    for i, name in enumerate(BAND_NAMES):
        band  = img[i]
        valid = band[band > 0]
        if len(valid) == 0:
            valid = np.array([0.0])
        feats[f'{name}_mean'] = float(valid.mean())
        feats[f'{name}_std']  = float(valid.std())
        feats[f'{name}_p10']  = float(np.percentile(valid, 10))
        feats[f'{name}_p90']  = float(np.percentile(valid, 90))

    red, nir, swir1, green = img[2], img[3], img[4], img[1]

    ndvi = np.where((nir + red) > 0, (nir - red) / (nir + red + 1e-8), 0)
    feats['NDVI_mean'] = float(ndvi.mean())
    feats['NDVI_std']  = float(ndvi.std())

    ndbi = np.where((swir1 + nir) > 0, (swir1 - nir) / (swir1 + nir + 1e-8), 0)
    feats['NDBI_mean'] = float(ndbi.mean())
    feats['NDBI_std']  = float(ndbi.std())

    mndwi = np.where((green + swir1) > 0, (green - swir1) / (green + swir1 + 1e-8), 0)
    feats['MNDWI_mean'] = float(mndwi.mean())

    feats['NIR_spatial_var'] = float(nir.var())
    feats['bright_frac'] = float((red > 0.15).mean())

    return feats


def build_feature_matrix(df, patch_dir, desc=''):
    rows, failed = [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        path = f"{patch_dir}/{row['patch_id']}.tif"
        try:
            rows.append(extract_engineered_features(path))
        except Exception:
            failed.append(row['patch_id'])
            rows.append({})
    if failed:
        print(f'  Warning: {len(failed)} patches failed')
    return pd.DataFrame(rows)


print('Extracting engineered features ...')
X_train_eng = build_feature_matrix(train_df, PATCH_DIR, desc='Train')
X_val_eng   = build_feature_matrix(val_df,   PATCH_DIR, desc='Val')
X_test_eng  = build_feature_matrix(test_df,  PATCH_DIR, desc='Test')

STRAT_FEATS = ['ntl', 'builtup', 'elevation']
for feat in STRAT_FEATS:
    X_train_eng[feat] = train_df[feat].values
    X_val_eng[feat]   = val_df[feat].values
    X_test_eng[feat]  = test_df[feat].values

col_medians = X_train_eng.median()
X_train_eng = X_train_eng.fillna(col_medians)
X_val_eng   = X_val_eng.fillna(col_medians)
X_test_eng  = X_test_eng.fillna(col_medians)

y_train = train_df['has_coverage'].values
y_val   = val_df['has_coverage'].values
y_test  = test_df['has_coverage'].values

print(f'Engineered features: {X_train_eng.shape}')

## Step 5B: MOSAIKS Random Convolutional Features (4096-d)

In [ ]:
class MOSAIKSFeaturizer:
    """Random Convolutional Features (Rolf et al. 2021)."""
    def __init__(self, n_features=4096, patch_size=3, n_channels=6, seed=42, filter_chunk=256):
        rng     = np.random.RandomState(seed)
        filters = rng.randn(n_features, n_channels, patch_size, patch_size).astype(np.float32)
        self.filters      = torch.from_numpy(filters).to(DEVICE)
        self.bias         = 1.0
        self.filter_chunk = filter_chunk

    def featurize_batch(self, img_batch):
        img_batch = img_batch.to(DEVICE)
        all_out   = []
        with torch.no_grad():
            for i in range(0, self.filters.shape[0], self.filter_chunk):
                chunk = self.filters[i : i + self.filter_chunk]
                out   = Func.conv2d(img_batch, chunk, padding=1)
                out   = torch.relu(out + self.bias)
                out   = out.mean(dim=[2, 3])
                all_out.append(out.cpu())
        return torch.cat(all_out, dim=1).numpy()


def extract_mosaiks(dataset, mosaiks_model, batch_size=64, desc=''):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    feats, labels = [], []
    for imgs, lbls in tqdm(loader, desc=desc):
        feats.append(mosaiks_model.featurize_batch(imgs))
        labels.extend(lbls.numpy())
    return np.concatenate(feats, axis=0), np.array(labels)


mosaiks = MOSAIKSFeaturizer(n_features=4096, filter_chunk=256)
print('MOSAIKS featurizer ready — extracting ...')

M_train, _ = extract_mosaiks(train_dataset, mosaiks, desc='MOSAIKS Train')
M_val,   _ = extract_mosaiks(val_dataset,   mosaiks, desc='MOSAIKS Val')
M_test,  _ = extract_mosaiks(test_dataset,  mosaiks, desc='MOSAIKS Test')
print(f'MOSAIKS feature shape: {M_train.shape}')

## Step 5C: ResNet-18 Frozen Features (512-d)

In [ ]:
class ResNet18Featurizer(nn.Module):
    """Frozen ResNet-18 with first conv adapted for 6-band HLS input."""
    def __init__(self, n_input_bands=6):
        super().__init__()
        resnet   = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        old_conv = resnet.conv1
        new_conv = nn.Conv2d(n_input_bands, 64, kernel_size=7,
                             stride=2, padding=3, bias=False)
        with torch.no_grad():
            new_conv.weight[:, :3] = old_conv.weight
            new_conv.weight[:, 3:] = (
                old_conv.weight.mean(dim=1, keepdim=True)
                .expand(-1, n_input_bands - 3, -1, -1)
            )
        resnet.conv1  = new_conv
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        self.feat_dim = 512
        for param in self.backbone.parameters():
            param.requires_grad = False

    def forward(self, x):
        with torch.no_grad():
            features = self.backbone(x)
        return features.squeeze(-1).squeeze(-1)


def extract_nn_features(dataset, featurizer, batch_size=64, desc=''):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    feats, labels = [], []
    featurizer.eval()
    for imgs, lbls in tqdm(loader, desc=desc):
        feats.append(featurizer(imgs.to(DEVICE)).cpu().numpy())
        labels.extend(lbls.numpy())
    return np.concatenate(feats, axis=0), np.array(labels)


resnet_featurizer = ResNet18Featurizer(n_input_bands=6).to(DEVICE).eval()
print(f'ResNet-18 ready — output dim: {resnet_featurizer.feat_dim}')

R_train, _ = extract_nn_features(train_dataset, resnet_featurizer, desc='ResNet Train')
R_val,   _ = extract_nn_features(val_dataset,   resnet_featurizer, desc='ResNet Val')
R_test,  _ = extract_nn_features(test_dataset,  resnet_featurizer, desc='ResNet Test')
print(f'ResNet-18 feature shape: {R_train.shape}')

del resnet_featurizer
torch.cuda.empty_cache()

## Step 5D: DINOv2 ViT-Base Frozen Features (768-d)

In [ ]:
class DINOv2Featurizer(nn.Module):
    """Frozen DINOv2 ViT-Base with RGB band selection from 6-band HLS."""
    def __init__(self, n_input_bands=6):
        super().__init__()
        self.dino     = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14',
                                       pretrained=True, verbose=False)
        self.rgb_idx  = [2, 1, 0]
        self.feat_dim = 768
        for param in self.dino.parameters():
            param.requires_grad = False

    def forward(self, x):
        x_rgb = x[:, self.rgb_idx, :, :]
        with torch.no_grad():
            return self.dino(x_rgb)


print('Loading DINOv2 ViT-Base ...')
dino_featurizer = DINOv2Featurizer(n_input_bands=6).to(DEVICE).eval()
print(f'DINOv2 ready — output dim: {dino_featurizer.feat_dim}')

D_train, _ = extract_nn_features(train_dataset, dino_featurizer, batch_size=32, desc='DINOv2 Train')
D_val,   _ = extract_nn_features(val_dataset,   dino_featurizer, batch_size=32, desc='DINOv2 Val')
D_test,  _ = extract_nn_features(test_dataset,  dino_featurizer, batch_size=32, desc='DINOv2 Test')
print(f'DINOv2 feature shape: {D_train.shape}')

del dino_featurizer
torch.cuda.empty_cache()

## Step 6A: Train & Evaluate — XGBoost (Engineered Features)

In [ ]:
# Check GPU for XGBoost
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                            capture_output=True, text=True)
    gpu_name = result.stdout.strip()
    xgb_device = 'cuda'
    print(f'GPU detected: {gpu_name} → using device="cuda"')
except Exception:
    xgb_device = 'cpu'
    print('No GPU detected → falling back to CPU')

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    early_stopping_rounds=30,
    device=xgb_device,
    random_state=42,
    n_jobs=-1,
)

xgb_model.fit(
    X_train_eng, y_train,
    eval_set=[(X_val_eng, y_val)],
    verbose=50
)
print(f'Best iteration: {xgb_model.best_iteration}')

In [ ]:
y_prob_xgb = xgb_model.predict_proba(X_test_eng)[:, 1]

# Val-optimal threshold
prec_v, rec_v, thr_v = precision_recall_curve(y_val, xgb_model.predict_proba(X_val_eng)[:, 1])
f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-8)
OPT_THR_XGB = float(thr_v[np.argmax(f1_v)])
y_pred_xgb = (y_prob_xgb >= OPT_THR_XGB).astype(int)

pr_auc_xgb = average_precision_score(y_test, y_prob_xgb)
auc_xgb    = roc_auc_score(y_test, y_prob_xgb)
f1_xgb     = f1_score(y_test, y_pred_xgb)
prec_xgb   = precision_score(y_test, y_pred_xgb, zero_division=0)
rec_xgb    = recall_score(y_test, y_pred_xgb)
acc_xgb    = accuracy_score(y_test, y_pred_xgb)

print('=== XGBoost (Engineered Features) ===')
print(f'PR-AUC  : {pr_auc_xgb:.4f}')
print(f'ROC-AUC : {auc_xgb:.4f}')
print(f'F1 @ thr={OPT_THR_XGB:.3f} : {f1_xgb:.4f}')
print(f'Precision: {prec_xgb:.4f}  Recall: {rec_xgb:.4f}  Accuracy: {acc_xgb:.4f}')
print()
print(classification_report(y_test, y_pred_xgb, target_names=['No Coverage', 'Has Coverage']))

# Diagnostic plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_test, y_pred_xgb)
im = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
fig.colorbar(im, ax=axes[0])
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['No Cov.', 'Has Cov.'])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(['No Cov.', 'Has Cov.'])
for r in range(2):
    for c in range(2):
        axes[0].text(c, r, str(cm[r, c]), ha='center', va='center',
                     color='white' if cm[r, c] > cm.max() / 2 else 'black', fontsize=14)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix')

prec_t, rec_t, _ = precision_recall_curve(y_test, y_prob_xgb)
axes[1].plot(rec_t, prec_t, lw=2, label=f'PR-AUC = {pr_auc_xgb:.3f}')
axes[1].axhline(y_test.mean(), linestyle='--', color='grey',
                label=f'Baseline ({y_test.mean():.3f})')
axes[1].scatter([rec_xgb], [prec_xgb], marker='*', s=200, color='red', zorder=5,
                label=f'Val-opt thr={OPT_THR_XGB:.3f}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='upper right'); axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1])

plt.suptitle('XGBoost (Engineered Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb19_xgboost_engineered_eval.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# XGBoost learning curve
results_xgb = xgb_model.evals_result()
val_auc_curve = results_xgb['validation_0']['auc']

plt.figure(figsize=(8, 4))
plt.plot(val_auc_curve, label='Val AUC')
plt.axvline(xgb_model.best_iteration, color='red', linestyle='--',
            label=f'Best ({xgb_model.best_iteration})')
plt.xlabel('Boosting Round'); plt.ylabel('AUC')
plt.title('XGBoost — Validation AUC Learning Curve')
plt.legend(); plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb19_xgboost_learning_curve.png', dpi=150)
plt.show()

In [ ]:
# Feature importance
feat_imp = pd.Series(
    xgb_model.feature_importances_,
    index=X_train_eng.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 7))
feat_imp.head(20).plot(kind='barh')
plt.xlabel('Feature Importance (gain)')
plt.title('Top 20 Features — XGBoost Engineered')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb19_xgboost_feature_importance.png', dpi=150)
plt.show()

## Step 6B: Train & Evaluate — MOSAIKS + RidgeClassifierCV

In [ ]:
M_trainval = np.concatenate([M_train, M_val], axis=0)
y_trainval = np.concatenate([y_train, y_val], axis=0)

ridge = RidgeClassifierCV(alphas=[0.01, 0.1, 1, 10, 100])
ridge.fit(M_trainval, y_trainval)
print(f'Best alpha: {ridge.alpha_}')

# RidgeClassifierCV uses decision_function scores (not calibrated probabilities)
y_scores_val_mosaiks  = ridge.decision_function(M_val)
y_scores_test_mosaiks = ridge.decision_function(M_test)

# Val-optimal threshold
prec_v, rec_v, thr_v = precision_recall_curve(y_val, y_scores_val_mosaiks)
f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-8)
OPT_THR_MOSAIKS = float(thr_v[np.argmax(f1_v)])
y_pred_mosaiks = (y_scores_test_mosaiks >= OPT_THR_MOSAIKS).astype(int)

pr_auc_mosaiks = average_precision_score(y_test, y_scores_test_mosaiks)
auc_mosaiks    = roc_auc_score(y_test, y_scores_test_mosaiks)
f1_mosaiks     = f1_score(y_test, y_pred_mosaiks)
prec_mosaiks   = precision_score(y_test, y_pred_mosaiks, zero_division=0)
rec_mosaiks    = recall_score(y_test, y_pred_mosaiks)
acc_mosaiks    = accuracy_score(y_test, y_pred_mosaiks)

print('=== MOSAIKS + Ridge ===')
print(f'PR-AUC  : {pr_auc_mosaiks:.4f}')
print(f'ROC-AUC : {auc_mosaiks:.4f}')
print(f'F1 @ thr={OPT_THR_MOSAIKS:.3f} : {f1_mosaiks:.4f}')
print(f'Precision: {prec_mosaiks:.4f}  Recall: {rec_mosaiks:.4f}  Accuracy: {acc_mosaiks:.4f}')
print()
print(classification_report(y_test, y_pred_mosaiks, target_names=['No Coverage', 'Has Coverage']))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_test, y_pred_mosaiks)
im = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
fig.colorbar(im, ax=axes[0])
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['No Cov.', 'Has Cov.'])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(['No Cov.', 'Has Cov.'])
for r in range(2):
    for c in range(2):
        axes[0].text(c, r, str(cm[r, c]), ha='center', va='center',
                     color='white' if cm[r, c] > cm.max() / 2 else 'black', fontsize=14)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix')

prec_t, rec_t, _ = precision_recall_curve(y_test, y_scores_test_mosaiks)
axes[1].plot(rec_t, prec_t, lw=2, label=f'PR-AUC = {pr_auc_mosaiks:.3f}')
axes[1].axhline(y_test.mean(), linestyle='--', color='grey',
                label=f'Baseline ({y_test.mean():.3f})')
axes[1].scatter([rec_mosaiks], [prec_mosaiks], marker='*', s=200, color='red', zorder=5,
                label=f'Val-opt thr={OPT_THR_MOSAIKS:.3f}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='upper right'); axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1])

plt.suptitle('MOSAIKS + Ridge', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb19_mosaiks_ridge_eval.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 6C: Train & Evaluate — ResNet-18 + LogisticRegressionCV

In [ ]:
R_trainval = np.concatenate([R_train, R_val], axis=0)

scaler_r       = StandardScaler()
R_trainval_sc  = scaler_r.fit_transform(R_trainval)
R_test_sc      = scaler_r.transform(R_test)
R_val_sc       = scaler_r.transform(R_val)

lr_resnet = LogisticRegressionCV(
    Cs=[0.01, 0.1, 1, 10],
    cv=3,
    max_iter=1000,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)
lr_resnet.fit(R_trainval_sc, y_trainval)
print(f'Best C: {lr_resnet.C_[0]:.4f}')

y_prob_resnet     = lr_resnet.predict_proba(R_test_sc)[:, 1]
y_val_prob_resnet = lr_resnet.predict_proba(R_val_sc)[:, 1]

prec_v, rec_v, thr_v = precision_recall_curve(y_val, y_val_prob_resnet)
f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-8)
OPT_THR_RESNET = float(thr_v[np.argmax(f1_v)])
y_pred_resnet = (y_prob_resnet >= OPT_THR_RESNET).astype(int)

pr_auc_resnet = average_precision_score(y_test, y_prob_resnet)
auc_resnet    = roc_auc_score(y_test, y_prob_resnet)
f1_resnet     = f1_score(y_test, y_pred_resnet)
prec_resnet   = precision_score(y_test, y_pred_resnet, zero_division=0)
rec_resnet    = recall_score(y_test, y_pred_resnet)
acc_resnet    = accuracy_score(y_test, y_pred_resnet)

print('=== ResNet-18 + Linear Probe ===')
print(f'PR-AUC  : {pr_auc_resnet:.4f}')
print(f'ROC-AUC : {auc_resnet:.4f}')
print(f'F1 @ thr={OPT_THR_RESNET:.3f} : {f1_resnet:.4f}')
print(f'Precision: {prec_resnet:.4f}  Recall: {rec_resnet:.4f}  Accuracy: {acc_resnet:.4f}')
print()
print(classification_report(y_test, y_pred_resnet, target_names=['No Coverage', 'Has Coverage']))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_test, y_pred_resnet)
im = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
fig.colorbar(im, ax=axes[0])
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['No Cov.', 'Has Cov.'])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(['No Cov.', 'Has Cov.'])
for r in range(2):
    for c in range(2):
        axes[0].text(c, r, str(cm[r, c]), ha='center', va='center',
                     color='white' if cm[r, c] > cm.max() / 2 else 'black', fontsize=14)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix')

prec_t, rec_t, _ = precision_recall_curve(y_test, y_prob_resnet)
axes[1].plot(rec_t, prec_t, lw=2, label=f'PR-AUC = {pr_auc_resnet:.3f}')
axes[1].axhline(y_test.mean(), linestyle='--', color='grey',
                label=f'Baseline ({y_test.mean():.3f})')
axes[1].scatter([rec_resnet], [prec_resnet], marker='*', s=200, color='red', zorder=5,
                label=f'Val-opt thr={OPT_THR_RESNET:.3f}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='upper right'); axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1])

plt.suptitle('ResNet-18 + LogisticRegressionCV', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb19_resnet18_linear_eval.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 6D: Train & Evaluate — DINOv2 + LogisticRegressionCV

In [ ]:
D_trainval = np.concatenate([D_train, D_val], axis=0)

scaler_d       = StandardScaler()
D_trainval_sc  = scaler_d.fit_transform(D_trainval)
D_test_sc      = scaler_d.transform(D_test)
D_val_sc       = scaler_d.transform(D_val)

lr_dino = LogisticRegressionCV(
    Cs=[0.01, 0.1, 1, 10],
    cv=3,
    max_iter=1000,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)
lr_dino.fit(D_trainval_sc, y_trainval)
print(f'Best C: {lr_dino.C_[0]:.4f}')

y_prob_dino     = lr_dino.predict_proba(D_test_sc)[:, 1]
y_val_prob_dino = lr_dino.predict_proba(D_val_sc)[:, 1]

prec_v, rec_v, thr_v = precision_recall_curve(y_val, y_val_prob_dino)
f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-8)
OPT_THR_DINO = float(thr_v[np.argmax(f1_v)])
y_pred_dino = (y_prob_dino >= OPT_THR_DINO).astype(int)

pr_auc_dino = average_precision_score(y_test, y_prob_dino)
auc_dino    = roc_auc_score(y_test, y_prob_dino)
f1_dino     = f1_score(y_test, y_pred_dino)
prec_dino   = precision_score(y_test, y_pred_dino, zero_division=0)
rec_dino    = recall_score(y_test, y_pred_dino)
acc_dino    = accuracy_score(y_test, y_pred_dino)

print('=== DINOv2 ViT-Base + Linear Probe ===')
print(f'PR-AUC  : {pr_auc_dino:.4f}')
print(f'ROC-AUC : {auc_dino:.4f}')
print(f'F1 @ thr={OPT_THR_DINO:.3f} : {f1_dino:.4f}')
print(f'Precision: {prec_dino:.4f}  Recall: {rec_dino:.4f}  Accuracy: {acc_dino:.4f}')
print()
print(classification_report(y_test, y_pred_dino, target_names=['No Coverage', 'Has Coverage']))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_test, y_pred_dino)
im = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
fig.colorbar(im, ax=axes[0])
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['No Cov.', 'Has Cov.'])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(['No Cov.', 'Has Cov.'])
for r in range(2):
    for c in range(2):
        axes[0].text(c, r, str(cm[r, c]), ha='center', va='center',
                     color='white' if cm[r, c] > cm.max() / 2 else 'black', fontsize=14)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix')

prec_t, rec_t, _ = precision_recall_curve(y_test, y_prob_dino)
axes[1].plot(rec_t, prec_t, lw=2, label=f'PR-AUC = {pr_auc_dino:.3f}')
axes[1].axhline(y_test.mean(), linestyle='--', color='grey',
                label=f'Baseline ({y_test.mean():.3f})')
axes[1].scatter([rec_dino], [prec_dino], marker='*', s=200, color='red', zorder=5,
                label=f'Val-opt thr={OPT_THR_DINO:.3f}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='upper right'); axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1])

plt.suptitle('DINOv2 ViT-Base + LogisticRegressionCV', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb19_dinov2_linear_eval.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 6E: Train & Evaluate — Prithvi-300M Linear Probe (Cached Embeddings)
Load frozen 1024-d Prithvi embeddings from `outputs/features/` (extracted by NB16).
Head-only classifier trained on pre-extracted embeddings — no encoder on GPU.

In [ ]:
OUTPUT_FEATURES = Path('outputs/features')
OUTPUT_FEATURES.mkdir(parents=True, exist_ok=True)


def load_or_extract_embeddings(split_name, df):
    """Load cached frozen Prithvi embeddings, or extract + cache them."""
    path = OUTPUT_FEATURES / f'prithvi_frozen_embeds_{split_name}.npz'
    if path.exists():
        data = np.load(path)
        X = data['X'].astype(np.float32)
        print(f'  Loaded cached {split_name}: {X.shape}')
        return X

    print(f'  Cache miss for {split_name} — extracting from scratch ...')
    from terratorch.registry import BACKBONE_REGISTRY

    encoder = BACKBONE_REGISTRY.build('prithvi_eo_v2_300', pretrained=True).to(DEVICE).eval()
    for p in encoder.parameters():
        p.requires_grad = False

    class _PrithviExtractDataset(Dataset):
        def __init__(self, df, patch_dir, n_bands=6):
            self.df = df.reset_index(drop=True)
            self.patch_dir = patch_dir
            self.n_bands = n_bands
        def __len__(self):
            return len(self.df)
        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            with rasterio.open(f"{self.patch_dir}/{row['patch_id']}.tif") as src:
                img = src.read(list(range(1, self.n_bands + 1))).astype(np.float32)
            img *= SCALE
            for b in range(self.n_bands):
                img[b] = (img[b] - HLS_MEANS[b]) / HLS_STDS[b]
            img = np.nan_to_num(img, nan=0.0)
            return torch.from_numpy(img).unsqueeze(1), row['patch_id']

    ds = _PrithviExtractDataset(df, PATCH_DIR)
    loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

    all_embs = []
    with torch.no_grad():
        for x, _ in tqdm(loader, desc=f'Extracting {split_name}'):
            x = x.to(DEVICE, dtype=torch.float32)
            feats = encoder(x)
            pooled = feats[-1].mean(dim=1)
            all_embs.append(pooled.cpu().numpy().astype(np.float32))

    X = np.concatenate(all_embs, axis=0)
    np.savez_compressed(path, X=X, patch_id=df['patch_id'].to_numpy())
    print(f'  Extracted and cached {split_name}: {X.shape}')

    del encoder
    torch.cuda.empty_cache()
    return X


print('Loading frozen Prithvi embeddings (1024-d) ...')
P_train_emb = load_or_extract_embeddings('train', train_df)
P_val_emb   = load_or_extract_embeddings('val',   val_df)
P_test_emb  = load_or_extract_embeddings('test',  test_df)
print(f'Train: {P_train_emb.shape}  |  Val: {P_val_emb.shape}  |  Test: {P_test_emb.shape}')


class EmbeddingClassDataset(Dataset):
    """Pre-extracted embeddings + integer class labels."""
    def __init__(self, embeddings, labels):
        self.X = torch.from_numpy(embeddings)
        self.y = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class PrithviClassifier(pl.LightningModule):
    """Head-only binary classifier on frozen Prithvi embeddings (1024-d)."""
    def __init__(self, lr=1e-3, num_classes=2):
        super().__init__()
        self.save_hyperparameters()

        self.head = nn.Sequential(
            nn.LayerNorm(1024),
            nn.Dropout(0.1),
            nn.Linear(1024, num_classes),
        )
        self.loss_fn = nn.CrossEntropyLoss()

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'Trainable params: {trainable:,}')

    def forward(self, x):
        """x: (B, 1024) pre-extracted embeddings → logits (B, num_classes)"""
        return self.head(x)

    def training_step(self, batch, batch_idx):
        x, y   = batch
        logits = self(x)
        loss   = self.loss_fn(logits, y)
        acc    = (logits.argmax(dim=1) == y).float().mean()
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_acc',  acc,  prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y   = batch
        logits = self(x)
        loss   = self.loss_fn(logits, y)
        preds  = logits.argmax(dim=1)
        acc    = (preds == y).float().mean()
        tp   = ((preds == 1) & (y == 1)).sum().float()
        pp   = (preds == 1).sum().float()
        ap   = (y == 1).sum().float()
        prec = tp / (pp + 1e-8)
        rec  = tp / (ap + 1e-8)
        f1   = 2 * prec * rec / (prec + rec + 1e-8)
        self.log('val_loss', loss, prog_bar=True, sync_dist=True)
        self.log('val_acc',  acc,  prog_bar=True, sync_dist=True)
        self.log('val_f1',   f1,   prog_bar=True, sync_dist=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.lr,
            weight_decay=0.0,
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=10
        )
        return {
            'optimizer': optimizer,
            'lr_scheduler': {'scheduler': scheduler, 'monitor': 'val_loss'},
        }

In [ ]:
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint


PRITHVI_BS = 32

# Weighted sampler on embedding dataset
labels_train = train_df['has_coverage'].values
class_counts  = np.bincount(labels_train)
class_weights = 1.0 / class_counts
sample_weights = class_weights[labels_train]
train_sampler = WeightedRandomSampler(
    weights=sample_weights, num_samples=len(labels_train), replacement=True
)

prithvi_train_loader = DataLoader(
    EmbeddingClassDataset(P_train_emb, y_train),
    batch_size=PRITHVI_BS, sampler=train_sampler,
    num_workers=0, pin_memory=True,
)
prithvi_val_loader = DataLoader(
    EmbeddingClassDataset(P_val_emb, y_val),
    batch_size=PRITHVI_BS * 2, shuffle=False,
    num_workers=0, pin_memory=True,
)

prithvi_model = PrithviClassifier(lr=1e-3)

early_stop = EarlyStopping(
    monitor='val_loss', patience=20, mode='min', verbose=True
)
checkpoint_cb = ModelCheckpoint(
    dirpath='checkpoints',
    filename='prithvi_linear_{epoch:02d}_{val_f1:.4f}',
    monitor='val_f1', mode='max', save_top_k=1, verbose=True
)

trainer = Trainer(
    max_epochs=100,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    callbacks=[early_stop, checkpoint_cb],
    log_every_n_steps=10,
    enable_progress_bar=True,
)

trainer.fit(prithvi_model, prithvi_train_loader, prithvi_val_loader)
print(f'Best checkpoint: {checkpoint_cb.best_model_path}')
print(f'Best val F1:     {checkpoint_cb.best_model_score:.4f}')

In [ ]:
# Load best checkpoint and evaluate
best_prithvi = PrithviClassifier.load_from_checkpoint(
    checkpoint_cb.best_model_path
).to(DEVICE).eval()

# Val inference → optimal threshold
prithvi_val_test_loader = DataLoader(
    EmbeddingClassDataset(P_val_emb, y_val),
    batch_size=256, shuffle=False, num_workers=0,
)
val_probs, val_labels = [], []
with torch.no_grad():
    for x, y in prithvi_val_test_loader:
        logits = best_prithvi(x.to(DEVICE))
        probs  = torch.softmax(logits, dim=1)[:, 1]
        val_probs.extend(probs.cpu().numpy())
        val_labels.extend(y.numpy())

y_val_prob_prithvi = np.array(val_probs)
y_val_true_prithvi = np.array(val_labels)

prec_v, rec_v, thr_v = precision_recall_curve(y_val_true_prithvi, y_val_prob_prithvi)
f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-8)
OPT_THR_PRITHVI = float(thr_v[np.argmax(f1_v)])
print(f'Prithvi optimal threshold (val): {OPT_THR_PRITHVI:.4f}')

# Test inference
prithvi_test_eval_loader = DataLoader(
    EmbeddingClassDataset(P_test_emb, y_test),
    batch_size=256, shuffle=False, num_workers=0,
)
test_probs, test_labels = [], []
with torch.no_grad():
    for x, y in prithvi_test_eval_loader:
        logits = best_prithvi(x.to(DEVICE))
        probs  = torch.softmax(logits, dim=1)[:, 1]
        test_probs.extend(probs.cpu().numpy())
        test_labels.extend(y.numpy())

y_prob_prithvi = np.array(test_probs)
y_pred_prithvi = (y_prob_prithvi >= OPT_THR_PRITHVI).astype(int)

pr_auc_prithvi = average_precision_score(y_test, y_prob_prithvi)
auc_prithvi    = roc_auc_score(y_test, y_prob_prithvi)
f1_prithvi     = f1_score(y_test, y_pred_prithvi)
prec_prithvi   = precision_score(y_test, y_pred_prithvi, zero_division=0)
rec_prithvi    = recall_score(y_test, y_pred_prithvi)
acc_prithvi    = accuracy_score(y_test, y_pred_prithvi)

print('=== Prithvi-300M Linear Probe ===')
print(f'PR-AUC  : {pr_auc_prithvi:.4f}')
print(f'ROC-AUC : {auc_prithvi:.4f}')
print(f'F1 @ thr={OPT_THR_PRITHVI:.3f} : {f1_prithvi:.4f}')
print(f'Precision: {prec_prithvi:.4f}  Recall: {rec_prithvi:.4f}  Accuracy: {acc_prithvi:.4f}')
print()
print(classification_report(y_test, y_pred_prithvi, target_names=['No Coverage', 'Has Coverage']))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_test, y_pred_prithvi)
im = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
fig.colorbar(im, ax=axes[0])
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['No Cov.', 'Has Cov.'])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(['No Cov.', 'Has Cov.'])
for r in range(2):
    for c in range(2):
        axes[0].text(c, r, str(cm[r, c]), ha='center', va='center',
                     color='white' if cm[r, c] > cm.max() / 2 else 'black', fontsize=14)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix')

prec_t, rec_t, _ = precision_recall_curve(y_test, y_prob_prithvi)
axes[1].plot(rec_t, prec_t, lw=2, label=f'PR-AUC = {pr_auc_prithvi:.3f}')
axes[1].axhline(y_test.mean(), linestyle='--', color='grey',
                label=f'Baseline ({y_test.mean():.3f})')
axes[1].scatter([rec_prithvi], [prec_prithvi], marker='*', s=200, color='red', zorder=5,
                label=f'Val-opt thr={OPT_THR_PRITHVI:.3f}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='upper right'); axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1])

plt.suptitle('Prithvi-300M Linear Probe', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb19_prithvi_linear_eval.png', dpi=150, bbox_inches='tight')
plt.show()

del best_prithvi
torch.cuda.empty_cache()

## Step 7: Cross-Model Comparison

In [ ]:
comparison_rows = [
    {'Model': 'XGBoost (Engineered)',   'PR-AUC': pr_auc_xgb,     'ROC-AUC': auc_xgb,
     'F1': f1_xgb,     'Precision': prec_xgb,     'Recall': rec_xgb,     'Accuracy': acc_xgb,
     'Threshold': OPT_THR_XGB},
    {'Model': 'MOSAIKS + Ridge',        'PR-AUC': pr_auc_mosaiks,  'ROC-AUC': auc_mosaiks,
     'F1': f1_mosaiks,  'Precision': prec_mosaiks,  'Recall': rec_mosaiks,  'Accuracy': acc_mosaiks,
     'Threshold': OPT_THR_MOSAIKS},
    {'Model': 'ResNet-18 + LogReg',     'PR-AUC': pr_auc_resnet,  'ROC-AUC': auc_resnet,
     'F1': f1_resnet,  'Precision': prec_resnet,  'Recall': rec_resnet,  'Accuracy': acc_resnet,
     'Threshold': OPT_THR_RESNET},
    {'Model': 'DINOv2 + LogReg',        'PR-AUC': pr_auc_dino,    'ROC-AUC': auc_dino,
     'F1': f1_dino,    'Precision': prec_dino,    'Recall': rec_dino,    'Accuracy': acc_dino,
     'Threshold': OPT_THR_DINO},
    {'Model': 'Prithvi-300M Linear',    'PR-AUC': pr_auc_prithvi, 'ROC-AUC': auc_prithvi,
     'F1': f1_prithvi, 'Precision': prec_prithvi, 'Recall': rec_prithvi, 'Accuracy': acc_prithvi,
     'Threshold': OPT_THR_PRITHVI},
]

comparison = pd.DataFrame(comparison_rows).sort_values('PR-AUC', ascending=False).reset_index(drop=True)

print('=== Phase 1 Binary Classification — NE India Test Set ===')
print(comparison.to_string(index=False))

comparison.to_csv(RESULTS_DIR / 'nb19_combined_binary_comparison.csv', index=False)
print(f'\nSaved: {RESULTS_DIR / "nb19_combined_binary_comparison.csv"}')

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(14, 5))

labels = comparison['Model'].tolist()
x = np.arange(len(labels))
w = 0.25

ax.bar(x - w, comparison['PR-AUC'],  w, label='PR-AUC',  color='#1f77b4')
ax.bar(x,     comparison['ROC-AUC'], w, label='ROC-AUC', color='#ff7f0e')
ax.bar(x + w, comparison['F1'],      w, label='F1',      color='#2ca02c')

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha='right')
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('Phase 1 Binary Classification — All Models\nNE India test set')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb19_all_models_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Overlaid PR curves for all models
fig, ax = plt.subplots(figsize=(9, 7))

models_pr = [
    ('XGBoost (Engineered)',  y_prob_xgb,             pr_auc_xgb),
    ('MOSAIKS + Ridge',      y_scores_test_mosaiks,   pr_auc_mosaiks),
    ('ResNet-18 + LogReg',   y_prob_resnet,           pr_auc_resnet),
    ('DINOv2 + LogReg',      y_prob_dino,             pr_auc_dino),
    ('Prithvi-300M Linear',  y_prob_prithvi,          pr_auc_prithvi),
]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for (name, scores, prauc), color in zip(models_pr, colors):
    prec_c, rec_c, _ = precision_recall_curve(y_test, scores)
    ax.plot(rec_c, prec_c, lw=2, color=color,
            label=f'{name} (PR-AUC={prauc:.3f})')

ax.axhline(y_test.mean(), linestyle='--', color='grey', alpha=0.6,
           label=f'Random baseline ({y_test.mean():.3f})')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves — All Phase 1 Models\nNE India test set', fontsize=13)
ax.legend(loc='upper right', fontsize=10)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb19_all_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8: Save Models & Push to GitHub

In [ ]:
import joblib

joblib.dump(xgb_model, MODELS_DIR / 'nb19_xgboost_engineered.pkl')
joblib.dump(ridge,     MODELS_DIR / 'nb19_mosaiks_ridge.pkl')
joblib.dump(lr_resnet, MODELS_DIR / 'nb19_resnet18_linear.pkl')
joblib.dump(lr_dino,   MODELS_DIR / 'nb19_dinov2_linear.pkl')
joblib.dump(scaler_r,  MODELS_DIR / 'nb19_resnet18_scaler.pkl')
joblib.dump(scaler_d,  MODELS_DIR / 'nb19_dinov2_scaler.pkl')

with open(MODELS_DIR / 'nb19_prithvi_linear_ckpt.txt', 'w') as f:
    f.write(checkpoint_cb.best_model_path)

# Per-model metrics CSVs
for row in comparison_rows:
    safe = row['Model'].lower().replace(' ', '_').replace('-', '').replace('(', '').replace(')', '').replace('+', '')
    pd.DataFrame([row]).to_csv(RESULTS_DIR / f'nb19_{safe}_metrics.csv', index=False)

print('All models and metrics saved.')

In [ ]:
import subprocess, os

token = "YOUR_TOKEN_HERE"
repo  = "tatyana21111/satellite-internet-prediction"

subprocess.run(['git', 'config', '--global', 'user.email', 'tatyana211zy@outlook.com'], check=True)
subprocess.run(['git', 'config', '--global', 'user.name',  'tatyana21111'], check=True)
subprocess.run(['git', 'remote', 'set-url', 'origin',
                f'https://{token}@github.com/{repo}.git'], check=True)

# Auto-discover output files
files = ['notebooks/02_binary_baselines.ipynb']
for pat in ['nb19_*']:
    files.extend([str(p) for p in FIGURES_DIR.glob(pat)])
    files.extend([str(p) for p in RESULTS_DIR.glob(pat)])
    files.extend([str(p) for p in MODELS_DIR.glob(pat)])

for f in files:
    if os.path.exists(f):
        subprocess.run(['git', 'add', '-f', f], check=True)
    else:
        print(f'Skipping (not yet generated): {f}')

diff = subprocess.run(['git', 'diff', '--cached', '--quiet'], capture_output=True)
if diff.returncode != 0:
    subprocess.run(['git', 'commit', '-m',
                    'NB02: Binary classification baselines (XGBoost + MOSAIKS + ResNet-18 + DINOv2 + Prithvi)'], check=True)
else:
    print('Nothing staged.')

subprocess.run(['git', 'pull', '--rebase', 'origin', 'main'], check=True)
subprocess.run(['git', 'push'], check=True)
print('Push successful.')